# 05 - Severity Distribution

**Purpose:** Identify the distribution that claim amounts follow, for each of the
four coverages. Candidate families are narrowed by the coefficient of variation,
skewness, and the mean excess plot, then ranked by AIC and BIC on maximum
likelihood fits, and the winner is confirmed with a QQ plot. No predictors are
involved; this is the marginal distribution only. Fitted on the claims table,
so the distribution is conditional on a claim having occurred.

**Input:** `data/processed/claims.csv`
**Output:** None. Conclusions carried into 07_SeverityModeling.
**Depends on:** 02_Cleaned_Data

**Candidates:** Gamma, Log-Normal, Weibull, Exponential, Pareto,
Inverse Gaussian



Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
PROC = ROOT / "data" / "processed"
FIGS = ROOT / "outputs" / "figures"

claims = pd.read_csv(PROC / "claims.csv")
claims = claims.rename(columns={'class': 'class_year'})

COVERAGES = ['Personal Property', 'Additional Living Expense',
             'Guest Medical', 'Liability']

sv = {c: claims[claims.coverage == c].reset_index(drop=True) for c in COVERAGES}

{c: len(d) for c, d in sv.items()}

Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

for ax, cov in zip(axes.flat, COVERAGES):
    d = sv[cov].amount
    d.hist(bins=50, ax=ax, color='#1f77b4')
    ax.set_title(f'{cov}  (n={len(d)})')
    ax.set_xlabel('claim amount ($)')
    ax.set_ylabel('claims')

plt.tight_layout()

Log-Histograms

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))

for ax, cov in zip(axes.flat, COVERAGES):
    d = sv[cov].amount
    np.log(d).hist(bins=50, ax=ax, color='#1f77b4')
    ax.set_title(f'{cov}  (n={len(d)})')
    ax.set_xlabel('log(claim amount)')
    ax.set_ylabel('claims')

plt.tight_layout()

Since these histograms all appear approximatley normal, we might think severity follows a lognormal distribution but further analysis of the BIC will be used to confirm

Our initial list of candidate distributions are Gamma, Log-Normal, Weibull, Exponenetial, inverse Gaussian, and Pareto


Starting With Personal Property

In [ ]:
from scipy import stats

x = sv['Personal Property'].amount.values
n = len(x)

results = {}

for name, dist, k in [('LogNormal',   stats.lognorm,      2),
                      ('Gamma',       stats.gamma,        2),
                      ('Weibull',     stats.weibull_min,  2),
                      ('InvGaussian', stats.invgauss,     2),
                      ('Pareto',      stats.pareto,       2),
                      ('Exponential', stats.expon,        1)]:
    par = dist.fit(x, floc=0)                 # location fixed at 0, not estimated
    ll  = dist.logpdf(x, *par).sum()
    results[name] = (ll, k)

tbl = pd.DataFrame(
    [(name, ll, k, 2*k - 2*ll, k*np.log(n) - 2*ll)
     for name, (ll, k) in results.items()],
    columns=['distribution', 'loglik', 'k', 'AIC', 'BIC']
).sort_values('BIC')

tbl.round(2)

*Thus we will assume severity of personal property claims follows a LogNormal distribution

Now to analyze additional living expense

In [ ]:
from scipy import stats

x = sv['Additional Living Expense'].amount.values
n = len(x)

results = {}

for name, dist, k in [('LogNormal',   stats.lognorm,      2),
                      ('Gamma',       stats.gamma,        2),
                      ('Weibull',     stats.weibull_min,  2),
                      ('InvGaussian', stats.invgauss,     2),
                      ('Pareto',      stats.pareto,       2),
                      ('Exponential', stats.expon,        1)]:
    par = dist.fit(x, floc=0)
    ll  = dist.logpdf(x, *par).sum()
    results[name] = (ll, k)

tbl = pd.DataFrame(
    [(name, ll, k, 2*k - 2*ll, k*np.log(n) - 2*ll)
     for name, (ll, k) in results.items()],
    columns=['distribution', 'loglik', 'k', 'AIC', 'BIC']
).sort_values('BIC')

tbl.round(2)

*Thus we will assume additional living expense severity follows lognormal

NOw to analyze Guest Medical

In [ ]:
from scipy import stats

x = sv['Guest Medical'].amount.values
n = len(x)

results = {}

for name, dist, k in [('LogNormal',   stats.lognorm,      2),
                      ('Gamma',       stats.gamma,        2),
                      ('Weibull',     stats.weibull_min,  2),
                      ('InvGaussian', stats.invgauss,     2),
                      ('Pareto',      stats.pareto,       2),
                      ('Exponential', stats.expon,        1)]:
    par = dist.fit(x, floc=0)
    ll  = dist.logpdf(x, *par).sum()
    results[name] = (ll, k)

tbl = pd.DataFrame(
    [(name, ll, k, 2*k - 2*ll, k*np.log(n) - 2*ll)
     for name, (ll, k) in results.items()],
    columns=['distribution', 'loglik', 'k', 'AIC', 'BIC']
).sort_values('BIC')

tbl.round(2)

*thus we will assume Guest medical loss severity follows a LogNormal distribution

Now to analyze Liability

In [ ]:
from scipy import stats

x = sv['Liability'].amount.values
n = len(x)

results = {}

for name, dist, k in [('LogNormal',   stats.lognorm,      2),
                      ('Gamma',       stats.gamma,        2),
                      ('Weibull',     stats.weibull_min,  2),
                      ('InvGaussian', stats.invgauss,     2),
                      ('Pareto',      stats.pareto,       2),
                      ('Exponential', stats.expon,        1)]:
    par = dist.fit(x, floc=0)
    ll  = dist.logpdf(x, *par).sum()
    results[name] = (ll, k)

tbl = pd.DataFrame(
    [(name, ll, k, 2*k - 2*ll, k*np.log(n) - 2*ll)
     for name, (ll, k) in results.items()],
    columns=['distribution', 'loglik', 'k', 'AIC', 'BIC']
).sort_values('BIC')

tbl.round(2)

Interestingly, Liabilty follows an exponential as opposed to lognormal, so we will move foward with that assumption